In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src import RDX, RDXConfig

In [ ]:
rng = np.random.default_rng(7)

# Three groups in 2D
cluster_1 = rng.normal(loc=[0.0, 0.0], scale=0.18, size=(12, 2))
cluster_2 = rng.normal(loc=[3.0, 0.0], scale=0.18, size=(12, 2))
cluster_3 = rng.normal(loc=[1.5, 2.5], scale=0.18, size=(12, 2))

X = np.vstack([cluster_1, cluster_2, cluster_3])
labels = np.array([0] * 12 + [1] * 12 + [2] * 12)

# Representation A: keep structure mostly intact
A = X.copy()

# Representation B: distort part of cluster_2 so neighborhood structure changes
B = X.copy()
B[12:18] += np.array([-1.0, 0.35])

A.shape, B.shape

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(A[:, 0], A[:, 1], c=labels)
axes[0].set_title("Representation A")

axes[1].scatter(B[:, 0], B[:, 1], c=labels)
axes[1].set_title("Representation B")

for ax in axes:
    ax.set_xlabel("dim 1")
    ax.set_ylabel("dim 2")

plt.tight_layout()
plt.show()

In [ ]:
config = RDXConfig(
    gamma=10.0,
    beta=5.0,
    num_explanations=3,
    explanation_size=5,
    random_state=0,
)

rdx = RDX(config)
result_ab, result_ba = rdx.fit_both_directions(A, B)

print("BSR(A, B):", result_ab.bsr)
print("BSR(B, A):", result_ba.bsr)

In [ ]:
for i, expl in enumerate(result_ab.explanations, start=1):
    print(f"RDX(A, B) Explanation {i}: {expl.indices}")

print()

for i, expl in enumerate(result_ba.explanations, start=1):
    print(f"RDX(B, A) Explanation {i}: {expl.indices}")

In [ ]:
def plot_explanations(points, explanations, title):
    plt.figure(figsize=(5, 4))
    plt.scatter(points[:, 0], points[:, 1], alpha=0.25)

    for idx, expl in enumerate(explanations, start=1):
        group = np.array(expl.indices)
        plt.scatter(points[group, 0], points[group, 1], s=80, label=f"Expl {idx}")

    plt.title(title)
    plt.xlabel("dim 1")
    plt.ylabel("dim 2")
    plt.legend()
    plt.show()

plot_explanations(A, result_ab.explanations, "RDX(A, B) shown in A")
plot_explanations(B, result_ab.explanations, "RDX(A, B) shown in B")
plot_explanations(B, result_ba.explanations, "RDX(B, A) shown in B")
plot_explanations(A, result_ba.explanations, "RDX(B, A) shown in A")

In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(result_ab.difference_matrix, aspect="auto")
plt.colorbar()
plt.title("Difference Matrix G(A, B)")
plt.show()

plt.figure(figsize=(5, 4))
plt.imshow(result_ab.affinity_matrix, aspect="auto")
plt.colorbar()
plt.title("Affinity Matrix F(A, B)")
plt.show()